# Leader-Follower Three-Arm Block-Attention Ablation

Single-source ablation testing whether restricting CD's cross-variate
attention to structure-aligned blocks changes the result. Three arms, one CSV.

**Experiment:** C=21, rho=0.5, phi=0.8, gamma in {0.0, 0.3, 0.6, 0.9},
seeds {42, 123, 456, 789, 1011}, modes {CI, CD, CD_Block}. 60 runs total.
Protocol is the committed leader-follower protocol verbatim:
schedule, early stopping, seeds, and per-mode batch sizes unchanged.
CD_Block runs at the CD-family committed batch 8.
DLinear is dropped: its committed gamma-sweep rows join no new contrast here.

**Models:** imported from the attached private Kaggle dataset `b1-block-attn`,
which must contain exactly `models.py` and `models_cd_block.py` copied
verbatim from this repository's revision branch. Tripwire asserts check
the file set, the AMP-fix marker, and `build_model` before anything runs.
CD_Block uses the pair-preserving partition (10 leader-follower pairs + the
isolate as a singleton), the one partition that lets block attention express
the lag-1 coupling.

**Instrumentation (near-zero marginal cost):** per-epoch mean pre-clip
gradient norm on every run; at gamma=0.6 additionally the participation ratio
of encoder-output feature covariance on a fixed 32-window validation probe,
via a forward hook. No attention weights are ever materialised.

**Outputs:** `/kaggle/working/results_block_attention.csv` (schema identical
to results_leader_follower.csv) and `/kaggle/working/diag_b5_gamma06.csv`
(per-epoch grad-norm rows for every run; participation-ratio column non-null
only in the gamma=0.6 cell that names the file).

**Engineering-spec deviation (declared):** resume is run-level, not
per-epoch. 60 independent runs, each short (committed-environment CD runs took
~13 min; treat ~25 min as an estimated ceiling), mean that a session death
costs at most one in-flight run; per-epoch optimiser/scaler/RNG
checkpointing is reserved for the multi-hour single runs it was
designed for.

**Estimated runtime:** ~9-12 h worst case on a T4. Dry-run measured lf_c21
CD at 38.1 s/epoch (batch 128); the committed protocol's CD and CD_Block at
batch 8 are unmeasured in this environment (the Stage 0 gap), so the upper
band is an extrapolation. Checkpoint after every run; safe to interrupt,
rerun Run All to resume.

**Launch:** new notebook from this file, GPU T4, attach dataset
`b1-block-attn`, Run All. Progress prints as `[n/60]` lines.


In [1]:
# ── Imports and device setup ──────────────────────────────────────────────────
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def free_cuda() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


Device: cuda
GPU:  Tesla T4
VRAM: 15.6 GB


In [2]:
# ── Committed-module import with tripwire asserts ─────────────────────────────
# Model code is NOT inlined. It comes from the attached private Kaggle dataset
# b1-block-attn, whose two files are verbatim copies from this
# repository's revision branch. The asserts fail fast if the wrong dataset, an
# extra file, or a pre-AMP-fix module version is attached.
import sys

EXPECTED_FILES = {"models.py", "models_cd_block.py"}
AMP_FIX_MARKER = "Allocated from the first encoder output"

INPUT_ROOT = Path("/kaggle/input/datasets/mendelowitzadi/b1-block-attn")
_hits = sorted(INPUT_ROOT.rglob("models_cd_block.py"))
assert len(_hits) == 1, (
    f"Expected exactly one attached dataset containing models_cd_block.py; found {_hits}. "
    "Attach b1-block-attn and nothing else that carries a models_cd_block.py."
)
MODULE_DIR = _hits[0].parent
_py_files = {p.name for p in MODULE_DIR.glob("*.py")}
assert _py_files == EXPECTED_FILES, (
    f"Dataset must contain exactly {sorted(EXPECTED_FILES)}; found {sorted(_py_files)}."
)
_src = (MODULE_DIR / "models_cd_block.py").read_text()
assert AMP_FIX_MARKER in _src, (
    "models_cd_block.py lacks the AMP-fix marker comment — a pre-fix version was uploaded. "
    "Re-copy the file from this repository's revision branch."
)

sys.path.insert(0, str(MODULE_DIR))
import models as M
import models_cd_block as MB

assert callable(getattr(M, "build_model", None)), "models.py must expose build_model()"
for _name in ("PatchTST_CD_Block", "leader_follower_groups"):
    assert hasattr(MB, _name), f"models_cd_block.py must expose {_name}"
import hashlib
for _f in sorted(EXPECTED_FILES):
    _h = hashlib.sha256((MODULE_DIR / _f).read_bytes()).hexdigest()[:16]
    print(f"  {_f}: sha256[:16]={_h}")
print(f"Committed modules loaded from {MODULE_DIR}")


  models.py: sha256[:16]=1eecbddbdf130528
  models_cd_block.py: sha256[:16]=8703aeed74877b1a
Committed modules loaded from /kaggle/input/datasets/mendelowitzadi/b1-block-attn


In [3]:
# ── Experiment configuration ──────────────────────────────────────────────────
# Committed leader-follower protocol verbatim. Changing any value
# breaks protocol identity with the committed CD/CI arms and the gamma=0
# sanity check against the AR(1) grid.

PHI:   float = 0.8
RHO:   float = 0.5
C:     int   = 21         # N_LEADERS + N_FOLLOWERS + N_ISOLATE

GAMMAS: list[float] = [0.0, 0.3, 0.6, 0.9]
SEEDS:  list[int]   = [42, 123, 456, 789, 1011]
MODES:  list[str]   = ["CI", "CD", "CD_Block"]
# DLinear dropped: committed rows join no new contrast.

# Sequence (must match results_leader_follower.csv exactly)
SEQ_LEN:      int = 512
PRED_LEN:     int = 96
PATCH_SIZE:   int = 16
PATCH_STRIDE: int = 8

# Architecture (Table 1, Synthetic + ETTh1 config)
D_MODEL:  int   = 64
N_HEADS:  int   = 8
N_LAYERS: int   = 3
DROPOUT:  float = 0.2

# Training
LR:             float = 1e-4
WEIGHT_DECAY:   float = 1e-4
MAX_EPOCHS:     int   = 50
PATIENCE:       int   = 10
GRAD_CLIP:      float = 1.0
WARMUP_EPOCHS:  int   = 10

# Batch sizes: CI at the committed CI batch; CD AND CD_Block at the CD-family
# committed batch 8. CD_Block could hold a far
# larger batch in this environment, but the ablation isolates attention scope,
# so both CD-family arms must share the identical protocol, step count
# included. The step-count asymmetry vs CI is the committed, disclosed one.
BATCH_BY_MODE: dict[str, int] = {"CI": 128, "CD": 8, "CD_Block": 8}

# Pair-preserving partition: 10 (leader, follower) groups + isolate singleton.
GROUPS: list[list[int]] = MB.leader_follower_groups()

# Instrumentation
B5_GAMMA:   float = 0.6   # participation ratio measured only in this cell
B5_PROBE_N: int   = 32    # fixed validation windows for the probe

N_PATCHES = (SEQ_LEN - PATCH_SIZE) // PATCH_STRIDE + 1
assert N_PATCHES == MB.num_patches(SEQ_LEN, PATCH_SIZE, PATCH_STRIDE)
print(f"N_PATCHES={N_PATCHES}  C*N={C * N_PATCHES}  "
      f"groups={len(GROUPS)} (sizes {sorted({len(g) for g in GROUPS})})  "
      f"total runs={len(GAMMAS) * len(SEEDS) * len(MODES)}")


N_PATCHES=63  C*N=1323  groups=11 (sizes [1, 2])  total runs=60


In [4]:
# ── Data generator (identical to the committed leader-follower notebook) ──────
N_LEADERS   = 10
N_FOLLOWERS = 10
N_ISOLATE   = 1
T_TOTAL     = 14_400
BURN_IN     = 1_000
TRAIN_FRAC  = 0.6
VAL_FRAC    = 0.2


def _make_transition(phi: float, gamma: float) -> np.ndarray:
    A = np.zeros((C, C), dtype=np.float64)
    np.fill_diagonal(A, phi)
    for k in range(N_FOLLOWERS):
        A[N_LEADERS + k, k] = gamma
    return A


def _make_cov(rho: float) -> np.ndarray:
    Sigma = np.full((C, C), rho, dtype=np.float64)
    np.fill_diagonal(Sigma, 1.0)
    if np.linalg.eigvalsh(Sigma).min() <= 0:
        raise ValueError(f"Covariance not PD for rho={rho}")
    return Sigma


def generate(gamma: float, rho: float, seed: int) -> np.ndarray:
    """VAR(1) leader-follower process. Returns (T_TOTAL, C) float64."""
    rng   = np.random.default_rng(seed)
    A     = _make_transition(PHI, gamma)
    L     = np.linalg.cholesky(_make_cov(rho))
    X     = np.zeros((T_TOTAL + BURN_IN, C), dtype=np.float64)
    for t in range(1, T_TOTAL + BURN_IN):
        X[t] = A @ X[t - 1] + L @ rng.standard_normal(C)
    return X[BURN_IN:]


def split_normalise(data: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    n_train = int(len(data) * TRAIN_FRAC)
    n_val   = int(len(data) * VAL_FRAC)
    train, val, test = data[:n_train], data[n_train:n_train+n_val], data[n_train+n_val:]
    mean = train.mean(axis=0, keepdims=True)
    std  = np.where(train.std(axis=0, keepdims=True) == 0, 1.0,
                    train.std(axis=0, keepdims=True))
    return (train-mean)/std, (val-mean)/std, (test-mean)/std


def make_windows(data: np.ndarray) -> tuple[torch.Tensor, torch.Tensor]:
    """Stride-tricks windowing — no Python loop."""
    T, Cv = data.shape
    n = T - SEQ_LEN - PRED_LEN + 1
    if n <= 0:
        raise ValueError(f"Not enough timesteps: {T}")
    s0, s1 = data.strides
    view = np.lib.stride_tricks.as_strided(
        data, shape=(n, SEQ_LEN + PRED_LEN, Cv), strides=(s0, s0, s1))
    xs = np.ascontiguousarray(view[:, :SEQ_LEN])
    ys = np.ascontiguousarray(view[:, SEQ_LEN:])
    return torch.tensor(xs, dtype=torch.float32), torch.tensor(ys, dtype=torch.float32)


# Quick window-count check
_n = int(T_TOTAL * TRAIN_FRAC) - SEQ_LEN - PRED_LEN + 1
print(f"Train windows: {_n}  "
      f"CI steps/epoch: {_n // 128 + (1 if _n % 128 else 0)}  "
      f"CD-family steps/epoch: {_n // 8 + (1 if _n % 8 else 0)}")


Train windows: 8033  CI steps/epoch: 63  CD-family steps/epoch: 1005


In [5]:
# ── Model construction from committed modules ─────────────────────────────────

def build_arm(mode: str) -> nn.Module:
    if mode in ("CI", "CD"):
        return M.build_model(
            mode, seq_len=SEQ_LEN, pred_len=PRED_LEN, num_variates=C,
            patch_size=PATCH_SIZE, stride=PATCH_STRIDE, d_model=D_MODEL,
            n_heads=N_HEADS, n_layers=N_LAYERS, dropout=DROPOUT)
    if mode == "CD_Block":
        return MB.PatchTST_CD_Block(
            groups=GROUPS, num_variates=C, seq_len=SEQ_LEN, pred_len=PRED_LEN,
            patch_size=PATCH_SIZE, stride=PATCH_STRIDE, d_model=D_MODEL,
            n_heads=N_HEADS, n_layers=N_LAYERS, dropout=DROPOUT)
    raise ValueError(f"Unknown mode: {mode}")


def _n_params(m: nn.Module) -> int:
    return sum(p.numel() for p in m.parameters())


# ── Architecture assertions ───────────────────────────────────────────────────
_x = torch.zeros(2, SEQ_LEN, C)
_models = {mode: build_arm(mode) for mode in MODES}
for _mode, _m in _models.items():
    _y = _m(_x)
    assert _y.shape == (2, PRED_LEN, C), f"{_mode} wrong output shape: {_y.shape}"
    assert _m.head.in_features == N_PATCHES * D_MODEL, f"{_mode} head wrong: {_m.head}"
    assert _m.head.out_features == PRED_LEN, f"{_mode} head wrong: {_m.head}"
    assert hasattr(_m, "encoder"), f"{_mode} lacks .encoder (hook target)"

# Equal capacity: any CD vs CD_Block difference must come from attention scope.
_p_cd, _p_blk = _n_params(_models["CD"]), _n_params(_models["CD_Block"])
assert _p_cd == _p_blk, f"Param mismatch: CD={_p_cd} CD_Block={_p_blk}"
assert _p_cd == _n_params(_models["CI"]), "CI/CD param mismatch"
print(f"All arms OK: {_p_cd} params each; heads Linear({N_PATCHES * D_MODEL}, {PRED_LEN})")
print(f"CD_Block partition: {GROUPS}")
del _models, _x, _y
free_cuda()


All arms OK: 538208 params each; heads Linear(4032, 96)
CD_Block partition: [[0, 10], [1, 11], [2, 12], [3, 13], [4, 14], [5, 15], [6, 16], [7, 17], [8, 18], [9, 19], [20]]


In [6]:
# ── Training engine (committed protocol + instrumentation) ────────────────────

def _cosine_warmup(optimizer, epoch: int, warmup: int) -> None:
    if epoch < warmup:
        lr = LR * (epoch + 1) / warmup
    else:
        progress = (epoch - warmup) / max(1, MAX_EPOCHS - warmup)
        lr = LR * 0.5 * (1.0 + np.cos(np.pi * progress))
    for g in optimizer.param_groups:
        g["lr"] = lr


@torch.no_grad()
def _evaluate(model: nn.Module, loader: DataLoader) -> tuple[float, float]:
    model.eval()
    mse = mae = n = 0.0
    for xb, yb in loader:
        pred = model(xb.to(DEVICE)).cpu()
        mse += nn.functional.mse_loss(pred, yb, reduction="sum").item()
        mae += nn.functional.l1_loss(pred,  yb, reduction="sum").item()
        n   += yb.numel()
    return mse / n, mae / n


def _participation_ratio(model: nn.Module, probe: torch.Tensor) -> float:
    """PR of encoder-output feature covariance on the fixed probe batch.

    A forward hook on the top-level encoder collects every encoder output of
    one probe forward (CD_Block calls its shared encoder once per group size,
    so all calls are concatenated), flattened to (tokens, D). PR =
    (sum lambda)^2 / sum lambda^2 of the D x D feature covariance: D means
    isotropic feature use, ~1 means collapse to one direction. Runs in eval
    mode outside autocast, so features are float32; no attention weights are
    materialised.
    """
    feats: list[torch.Tensor] = []

    def _hook(_mod, _inp, out):
        feats.append(out.detach().float().reshape(-1, out.shape[-1]))

    handle = model.encoder.register_forward_hook(_hook)
    was_training = model.training
    model.eval()
    try:
        with torch.no_grad():
            model(probe.to(DEVICE))
    finally:
        handle.remove()
        if was_training:
            model.train()
    F = torch.cat(feats, dim=0)
    F = F - F.mean(dim=0, keepdim=True)
    cov = (F.T @ F) / max(1, F.shape[0] - 1)
    ev  = torch.clamp(torch.linalg.eigvalsh(cov), min=0.0)
    s1, s2 = ev.sum().item(), (ev * ev).sum().item()
    return (s1 * s1) / s2 if s2 > 0.0 else float("nan")


def _fit(mode: str, gamma: float, seed: int,
         datasets: tuple, batch_size: int) -> tuple[dict, list[dict]]:
    x_tr, y_tr, x_va, y_va, x_te, y_te = datasets
    train_dl = DataLoader(TensorDataset(x_tr, y_tr), batch_size=batch_size,
                          shuffle=True,  drop_last=False)
    val_dl   = DataLoader(TensorDataset(x_va, y_va), batch_size=batch_size,
                          shuffle=False, drop_last=False)
    test_dl  = DataLoader(TensorDataset(x_te, y_te), batch_size=batch_size,
                          shuffle=False, drop_last=False)

    measure_pr = bool(np.isclose(gamma, B5_GAMMA))
    if measure_pr:
        assert len(x_va) >= B5_PROBE_N, f"Probe needs {B5_PROBE_N} val windows, have {len(x_va)}"
    probe = x_va[:B5_PROBE_N] if measure_pr else None

    model     = build_arm(mode).to(DEVICE)
    opt       = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler    = GradScaler("cuda", enabled=(DEVICE.type == "cuda"))
    criterion = nn.MSELoss()

    spe   = len(train_dl)   # steps per epoch
    best_val, best_epoch, best_steps = float("inf"), 0, 0
    best_state, no_improve, total    = None, 0, 0
    diag: list[dict] = []

    try:
        for epoch in range(MAX_EPOCHS):
            _cosine_warmup(opt, epoch, WARMUP_EPOCHS)
            model.train()
            gn_sum, gn_n, gn_bad = 0.0, 0, 0
            for xb, yb in train_dl:
                opt.zero_grad(set_to_none=True)
                with autocast("cuda", enabled=(DEVICE.type == "cuda")):
                    loss = criterion(model(xb.to(DEVICE)), yb.to(DEVICE))
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                # clip_grad_norm_ returns the PRE-clip total norm on unscaled grads.
                norm = nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP).item()
                if np.isfinite(norm):
                    gn_sum += norm
                    gn_n   += 1
                else:            # AMP overflow step: scaler skips it; excluded from mean
                    gn_bad += 1
                scaler.step(opt)
                scaler.update()
                total += 1

            val_mse, _ = _evaluate(model, val_dl)
            diag.append({
                "mode": mode, "gamma": gamma, "seed": seed, "epoch": epoch + 1,
                "grad_norm_mean": gn_sum / gn_n if gn_n else float("nan"),
                "nonfinite_steps": gn_bad,
                "participation_ratio":
                    _participation_ratio(model, probe) if measure_pr else float("nan"),
                "val_mse": val_mse,
            })
            if val_mse < best_val:
                best_val   = val_mse
                best_epoch = epoch + 1
                best_steps = total
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= PATIENCE:
                    break
    finally:
        if best_state:
            model.load_state_dict(best_state)
        test_mse, test_mae = _evaluate(model, test_dl)
        del model, opt, scaler
        free_cuda()

    row = {"dataset": "leader_follower_var1", "C": C, "rho": RHO, "gamma": gamma,
           "mode": mode, "seed": seed, "test_mse": test_mse, "test_mae": test_mae,
           "best_epoch": best_epoch, "batch_size": batch_size,
           "steps_per_epoch": spe, "total_steps": best_steps}
    return row, diag


def train_one(mode: str, gamma: float, seed: int) -> tuple[dict, list[dict]]:
    """Generate data once, then fit with OOM-safe batch halving."""
    set_seed(seed)
    raw = generate(gamma, RHO, seed)
    tr, va, te = split_normalise(raw)
    datasets   = (*make_windows(tr), *make_windows(va), *make_windows(te))
    batch_size = BATCH_BY_MODE[mode]

    while True:
        try:
            set_seed(seed)   # identical init on every OOM retry
            return _fit(mode, gamma, seed, datasets, batch_size)
        except RuntimeError as exc:
            if "out of memory" not in str(exc).lower():
                raise
            free_cuda()
            if batch_size == 1:
                raise
            batch_size = max(1, batch_size // 2)
            print(f"  OOM: retrying at batch_size={batch_size}")


In [7]:
# ── Main sweep with atomic checkpointing and safe resume ─────────────────────
OUT      = Path("/kaggle/working/results_block_attention.csv")
DIAG_OUT = Path("/kaggle/working/diag_b5_gamma06.csv")


def _key(gamma: float, mode: str, seed: int) -> tuple:
    return (round(float(gamma), 4), str(mode), int(seed))


def _save_atomic(rows: list[dict], path: Path) -> None:
    tmp = path.with_suffix(".csv.tmp")
    pd.DataFrame(rows).to_csv(tmp, index=False)
    os.replace(tmp, path)


if OUT.exists() and OUT.stat().st_size > 100:
    _existing = pd.read_csv(OUT)
    done      = {_key(r.gamma, r.mode, r.seed) for r in _existing.itertuples()}
    results   = _existing.to_dict("records")
    print(f"Resuming: {len(done)}/{len(GAMMAS)*len(MODES)*len(SEEDS)} runs done.")
else:
    done, results = set(), []

# Diag rows survive only for completed runs: rows from a killed in-flight run
# are pruned so a resumed run cannot leave duplicate epochs behind.
if DIAG_OUT.exists() and DIAG_OUT.stat().st_size > 100:
    _d       = pd.read_csv(DIAG_OUT)
    diag_all = [r for r in _d.to_dict("records")
                if _key(r["gamma"], r["mode"], r["seed"]) in done]
    if len(diag_all) != len(_d):
        print(f"Pruned {len(_d) - len(diag_all)} in-flight diag rows.")
        _save_atomic(diag_all, DIAG_OUT)
else:
    diag_all = []

total = len(GAMMAS) * len(MODES) * len(SEEDS)
idx   = 0
fails = []

for gamma in GAMMAS:
    for mode in MODES:
        for seed in SEEDS:
            idx += 1
            key = _key(gamma, mode, seed)
            if key in done:
                print(f"[{idx}/{total}] SKIP gamma={gamma} mode={mode} seed={seed}")
                continue
            print(f"[{idx}/{total}] gamma={gamma} mode={mode} seed={seed} ...",
                  end=" ", flush=True)
            t0 = time.time()
            try:
                row, diag = train_one(mode, gamma, seed)
            except Exception as exc:
                free_cuda()
                print(f"FAILED: {type(exc).__name__}: {exc}")
                fails.append((gamma, mode, seed, repr(exc)))
                continue
            print(f"mse={row['test_mse']:.4f}  epoch={row['best_epoch']}"
                  f"  bs={row['batch_size']}  ({time.time()-t0:.0f}s)")
            results.append(row)
            diag_all.extend(diag)
            done.add(key)
            # Diag first: a death between the two saves leaves orphan diag rows
            # (pruned on resume), never a completed result with lost diagnostics.
            _save_atomic(diag_all, DIAG_OUT)
            _save_atomic(results, OUT)

print(f"\nDone. {len(results)}/{total} runs -> {OUT}")
print(f"Diag rows: {len(diag_all)} -> {DIAG_OUT}")
if fails:
    print(f"{len(fails)} failures (rerun this cell to retry):")
    for f in fails:
        print(" ", f)


[1/60] gamma=0.0 mode=CI seed=42 ... mse=0.9626  epoch=22  bs=128  (309s)
[2/60] gamma=0.0 mode=CI seed=123 ... mse=1.0102  epoch=31  bs=128  (394s)
[3/60] gamma=0.0 mode=CI seed=456 ... mse=1.0543  epoch=28  bs=128  (365s)
[4/60] gamma=0.0 mode=CI seed=789 ... mse=1.0713  epoch=27  bs=128  (356s)
[5/60] gamma=0.0 mode=CI seed=1011 ... mse=1.0367  epoch=20  bs=128  (289s)
[6/60] gamma=0.0 mode=CD seed=42 ... mse=0.9643  epoch=7  bs=8  (822s)
[7/60] gamma=0.0 mode=CD seed=123 ... mse=1.0105  epoch=7  bs=8  (821s)
[8/60] gamma=0.0 mode=CD seed=456 ... mse=1.0587  epoch=7  bs=8  (820s)
[9/60] gamma=0.0 mode=CD seed=789 ... mse=1.0682  epoch=7  bs=8  (820s)
[10/60] gamma=0.0 mode=CD seed=1011 ... mse=1.0418  epoch=7  bs=8  (820s)
[11/60] gamma=0.0 mode=CD_Block seed=42 ... mse=0.9645  epoch=7  bs=8  (340s)
[12/60] gamma=0.0 mode=CD_Block seed=123 ... mse=1.0100  epoch=7  bs=8  (340s)
[13/60] gamma=0.0 mode=CD_Block seed=456 ... mse=1.0537  epoch=7  bs=8  (340s)
[14/60] gamma=0.0 mode=CD_Bl

In [8]:
# ── Summary: three-arm pivot (paste this cell's output) ──────────────────────
df = pd.read_csv(OUT)
print(f"Rows: {len(df)}  |  seeds: {sorted(df['seed'].unique())}  "
      f"|  modes: {sorted(df['mode'].unique())}")
print(f"Duplicates: {df.duplicated(['gamma','mode','seed']).sum()}")

piv = df.groupby(["gamma", "mode"])["test_mse"].agg(["mean", "std"]).unstack("mode")
piv.columns = [f"{stat}_{mode}" for stat, mode in piv.columns]

print("\n=== Three-arm pivot (mean ± std over seeds) ===")
print(f"{'gamma':>6} {'CI':>8} {'CD':>8} {'CD_Blk':>8} {'CD/CI':>7} {'Blk/CI':>7} {'Blk/CD':>7}")
for g in sorted(df["gamma"].unique()):
    ci  = piv.loc[g, "mean_CI"]
    cd  = piv.loc[g, "mean_CD"]
    blk = piv.loc[g, "mean_CD_Block"]
    print(f"{g:>6.1f} {ci:>8.4f} {cd:>8.4f} {blk:>8.4f} "
          f"{cd/ci:>7.4f} {blk/ci:>7.4f} {blk/cd:>7.4f}")

if 0.0 in set(np.round(df["gamma"], 4)):
    print("    (gamma=0 is the internal control: no coupling exists, so all "
          "three ratios should sit near 1; drift there indicates protocol error.)")

print("\n=== std by (gamma, mode) ===")
print(df.groupby(["gamma", "mode"])["test_mse"].std().round(4).unstack("mode").to_string())

# Diagnostics coverage
if DIAG_OUT.exists():
    dd = pd.read_csv(DIAG_OUT)
    pr = dd[dd["participation_ratio"].notna()]
    print(f"\n{len(dd)} diag rows; PR measured on {pr[['mode','seed']].drop_duplicates().shape[0]} "
          f"runs (expected {len(MODES) * len(SEEDS)} at gamma={B5_GAMMA}); "
          f"nonfinite grad steps total: {int(dd['nonfinite_steps'].sum())}")
else:
    print(f"\n{DIAG_OUT} not found — no completed runs yet.")

# Cross-environment reproducibility note (report-only, never pooled):
# if the committed leader-follower CSV is attached, print gamma-wise CI/CD
# means from both environments side by side.
for cand in sorted(Path("/kaggle/input").glob("*/results_leader_follower.csv")):
    old = pd.read_csv(cand)
    print("\n=== Committed-environment lf means (reproducibility reference only) ===")
    print(old[old["mode"].isin(["CI", "CD"])]
          .groupby(["gamma", "mode"])["test_mse"].mean().round(4).unstack("mode").to_string())
    break


Rows: 60  |  seeds: [np.int64(42), np.int64(123), np.int64(456), np.int64(789), np.int64(1011)]  |  modes: ['CD', 'CD_Block', 'CI']
Duplicates: 0

=== Three-arm pivot (mean ± std over seeds) ===
 gamma       CI       CD   CD_Blk   CD/CI  Blk/CI  Blk/CD
   0.0   1.0270   1.0287   1.0273  1.0016  1.0003  0.9986
   0.3   1.0326   1.0393   1.0372  1.0065  1.0045  0.9980
   0.6   1.0298   1.0377   1.0353  1.0077  1.0054  0.9977
   0.9   1.0278   1.0362   1.0335  1.0082  1.0056  0.9974
    (gamma=0 is the internal control: no coupling exists, so all three ratios should sit near 1; drift there indicates protocol error.)

=== std by (gamma, mode) ===
mode       CD  CD_Block      CI
gamma                          
0.0    0.0422    0.0411  0.0425
0.3    0.0656    0.0655  0.0623
0.6    0.0669    0.0677  0.0647
0.9    0.0660    0.0672  0.0645

B5: 1248 diag rows; PR measured on 15 runs (expected 15 at gamma=0.6); nonfinite grad steps total: 216


In [9]:
from IPython.display import FileLink
display(FileLink(str(OUT)))
display(FileLink(str(DIAG_OUT)))


/kaggle/working/results_block_attention.csv

/kaggle/working/diag_b5_gamma06.csv